# 📐 06 · Edges, Lines, and Curves
### Gradients, the Canny pipeline, the Hough transform, and Steger's curvilinear detector

**Part C · Finding structure** · Module 06 of 29 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/06_edges_lines_and_curves.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, SciPy |
| **Prerequisites** | Modules 02–04 |
| **Estimated time** | 90–105 minutes |
| **02-740 topics covered** | Image gradients; Roberts/Sobel/Prewitt/Kirsch operators; the Canny edge-detection pipeline (non-maximum suppression, hysteresis thresholding); the Hough transform for lines/curves; Steger's curvilinear-structure detection algorithm |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every filament/line image below is SIMULATED; nothing is read from disk.

## 🎯 Learning objectives

1. Compute image gradients and explain what edge strength and edge direction mean.
2. Compare the Roberts, Sobel, Prewitt, and Kirsch edge operators and explain their key differences.
3. Implement the full Canny pipeline: smoothing, gradient, non-maximum suppression, and hysteresis
   thresholding.
4. Implement a Hough transform for line detection from scratch and explain the voting idea.
5. Explain the basic idea of Steger's algorithm for detecting thin, curvilinear structures on
   grayscale images directly, without a prior binarization step.

## 🗺️ Contents

1. What is an edge, and how do you calculate a gradient?
2. Edge operators: Roberts, Sobel, Prewitt, Kirsch
3. The full Canny pipeline
4. The Hough transform for line (and curve) detection
5. Steger's algorithm for curvilinear structures

## 🧫 The biological question

Many of the most important structures in cell biology are not blobs or points but **extended, thin,
line-like or curved objects**: actin filaments and microtubules forming the cytoskeleton, the edges
of a migrating cell's leading edge, chromosome boundaries, and the walls of tubular organelles. A
point-detection pipeline (Module 05) is the wrong tool entirely for these structures — what is
needed instead is the ability to find sharp intensity transitions (edges) and trace extended,
possibly curved, line-like structures through an image, even when they are thin, noisy, and only a
few pixels wide.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 6
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)


SOBEL_Y = SOBEL_X.T


def sobel_gradients(image: np.ndarray):
    """Image gradients by the Sobel operator: returns (gx, gy, magnitude, orientation_radians)."""
    gx = ndi.correlate(image, SOBEL_X, mode="reflect")
    gy = ndi.correlate(image, SOBEL_Y, mode="reflect")
    mag = np.hypot(gx, gy)
    ori = np.arctan2(gy, gx)
    return gx, gy, mag, ori


def non_max_suppression_edges(mag: np.ndarray, ori: np.ndarray) -> np.ndarray:
    """Thin a gradient-magnitude image to 1-pixel-wide ridges along the gradient direction
    (the Canny edge detector's non-maximum-suppression step)."""
    angle = (np.rad2deg(ori) % 180)
    out = np.zeros_like(mag)
    padded = np.pad(mag, 1, mode="edge")
    h, w = mag.shape
    for dy, dx, lo, hi in ((0, 1, -22.5, 22.5), (1, 1, 22.5, 67.5),
                           (1, 0, 67.5, 112.5), (1, -1, 112.5, 157.5)):
        sel = ((angle >= lo) & (angle < hi)) if lo >= 0 else ((angle >= lo + 180) | (angle < hi))
        a = padded[1 + dy:1 + dy + h, 1 + dx:1 + dx + w]
        b = padded[1 - dy:1 - dy + h, 1 - dx:1 - dx + w]
        keep = sel & (mag >= a) & (mag >= b)
        out[keep] = mag[keep]
    return out


def hysteresis_threshold(mag: np.ndarray, low: float, high: float) -> np.ndarray:
    """Canny's hysteresis thresholding: keep strong edges (> high) and weak edges (> low) that
    are 8-connected to a strong edge."""
    strong = mag > high
    weak = (mag > low) & ~strong
    labeled, n = ndi.label(strong | weak, structure=np.ones((3, 3)))
    keep = np.zeros_like(strong)
    strong_labels = set(labeled[strong])
    for lbl in strong_labels:
        if lbl:
            keep |= labeled == lbl
    return keep


def hough_lines(edge_image: np.ndarray, n_theta: int = 180, n_rho: int = 200):
    """A from-scratch Hough transform for lines: returns the (rho, theta) accumulator and its
    rho/theta bin edges, given a binary edge image."""
    h, w = edge_image.shape
    diag = np.hypot(h, w)
    thetas = np.linspace(-np.pi / 2, np.pi / 2, n_theta)
    rhos = np.linspace(-diag, diag, n_rho)
    acc = np.zeros((n_rho, n_theta), dtype=int)
    ys, xs = np.nonzero(edge_image)
    cos_t, sin_t = np.cos(thetas), np.sin(thetas)
    for y, x in zip(ys, xs):
        rho_vals = x * cos_t + y * sin_t
        bins = np.clip(((rho_vals + diag) / (2 * diag) * (n_rho - 1)).astype(int), 0, n_rho - 1)
        acc[bins, np.arange(n_theta)] += 1
    return acc, rhos, thetas


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---
## 1 · What is an edge, and how do you calculate a gradient?

An edge (or edge point) is a pixel at or around which image intensity undergoes a sharp change.
Examined along the direction perpendicular (normal) to the edge, intensity looks like a 1-D step,
ramp, or roof signal; examined along the edge's own direction (tangent), intensity barely changes at
all. This directional asymmetry is exactly what a **gradient** captures: the gradient vector
$\nabla I = (I_x, I_y)$ points in the direction of steepest intensity increase, with **edge strength**
$\|\nabla I\| = \sqrt{I_x^2 + I_y^2}$ and **edge direction** $\theta = \arctan(I_y / I_x)$.

In [ ]:
# ── 1.1 build a synthetic filament network (think: actin/microtubules) ──
def simulate_filament_network(size=160, n_filaments=10, seed=0):
    rng_f = np.random.default_rng(seed)
    canvas = np.full((size, size), 0.05)
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
    for _ in range(n_filaments):
        cy, cx = rng_f.uniform(size * 0.1, size * 0.9, size=2)
        angle = rng_f.uniform(0, np.pi)
        length = rng_f.uniform(size * 0.25, size * 0.5)
        curvature = rng_f.uniform(-0.02, 0.02)
        n_samples = 200
        t = np.linspace(-length / 2, length / 2, n_samples)
        path_y = cy + t * np.sin(angle) + curvature * t ** 2 * np.cos(angle)
        path_x = cx + t * np.cos(angle) - curvature * t ** 2 * np.sin(angle)
        for py, px in zip(path_y, path_x):
            d2 = (yy - py) ** 2 + (xx - px) ** 2
            canvas += np.exp(-d2 / (2 * 1.0 ** 2)) * 0.15
    return np.clip(canvas, 0, None)

filament_clean = simulate_filament_network(seed=SEED)
filament_image = add_gaussian_noise(blur_image(filament_clean, gaussian_psf(1.0)), 0.025, rng)
show_image_grid([filament_clean, filament_image], ['clean synthetic filament network', 'blurred + noisy (what a microscope gives you)'], ncols=2, figsize_scale=4)
plt.show()

🔍 **What just happened?** We built a small inline simulator (not part of the shared toolkit,
since it is specific to this module) that draws curved filament-like paths — a stand-in for an actin
or microtubule network — then degraded it with the same blur-plus-noise model used throughout this
course. This gives us a controlled test case with a known underlying structure, so we can judge
later whether our edge- and line-detection pipeline is finding the real filaments.

In [ ]:
# ── 1.2 compute the gradient and visualize edge strength and direction ──
gx, gy, mag, ori = sobel_gradients(filament_image)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
show_image(axes[0], filament_image, 'image')
show_image(axes[1], mag, 'edge strength |grad I|', cmap='inferno')
axes[2].imshow(ori, cmap='hsv'); axes[2].set_title('edge direction (radians)', fontsize=10); axes[2].axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The edge-strength map lights up exactly along the filaments (where
intensity changes sharply across the filament's width) and stays dark in smooth background regions.
The direction map (color-coded by angle) is most informative right where edge strength is high — in
smooth regions, "direction" is essentially meaningless noise, since there is no real edge to have a
direction.

---
## 2 · Edge operators: Roberts, Sobel, Prewitt, Kirsch

Several small, hand-designed convolution kernels approximate the image gradient, each with
slightly different properties:

- **Roberts cross**: the oldest (1963), uses tiny $2\times2$ diagonal kernels
  $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ and $\begin{pmatrix}0&1\\-1&0\end{pmatrix}$ — fast but
  very sensitive to noise, since it does no smoothing at all.
- **Sobel**: $3\times3$, separable into an averaging kernel times a differencing kernel (e.g.
  $[1,2,1]^T * [-1,0,1]$), so it computes the gradient *with built-in smoothing* — this is why it is
  far more robust to noise than Roberts.
- **Prewitt**: very similar to Sobel but uses a uniform $[1,1,1]^T$ averaging kernel instead of the
  weighted $[1,2,1]^T$, giving equal weight to every row/column rather than emphasizing the center.
- **Kirsch**: a non-linear "compass" operator — one kernel is rotated through all 8 compass
  directions, and the edge magnitude is the *maximum* response across all 8, with the edge direction
  given by whichever rotation produced that maximum.

In [ ]:
# ── 2.1 compare all four operators on the same patch ──
ROBERTS_1 = np.array([[1, 0], [0, -1]], dtype=float)
ROBERTS_2 = np.array([[0, 1], [-1, 0]], dtype=float)
PREWITT_X = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=float)
PREWITT_Y = PREWITT_X.T

def roberts_magnitude(image):
    gx = ndi.correlate(image, ROBERTS_1, mode='reflect')
    gy = ndi.correlate(image, ROBERTS_2, mode='reflect')
    return np.hypot(gx, gy)

def prewitt_magnitude(image):
    gx = ndi.correlate(image, PREWITT_X, mode='reflect')
    gy = ndi.correlate(image, PREWITT_Y, mode='reflect')
    return np.hypot(gx, gy)

def kirsch_magnitude(image):
    base = np.array([[5, 5, 5], [-3, 0, -3], [-3, -3, -3]], dtype=float)
    responses = []
    k = base.copy()
    for _ in range(8):
        responses.append(ndi.correlate(image, k, mode='reflect'))
        k = np.rot90(k)   # rotate the compass kernel by 45 degrees each step (approximated via 90 degree grid rotation)
    return np.max(responses, axis=0)

_, _, sobel_mag, _ = sobel_gradients(filament_image)
roberts_mag = roberts_magnitude(filament_image)
prewitt_mag = prewitt_magnitude(filament_image)
kirsch_mag = kirsch_magnitude(filament_image)

show_image_grid([roberts_mag, sobel_mag, prewitt_mag, kirsch_mag],
                 ['Roberts (no smoothing)', 'Sobel', 'Prewitt', 'Kirsch (8-direction max)'], ncols=4)
plt.show()

🔍 **What just happened?** Roberts, having no built-in smoothing, shows the noisiest, "grainiest"
edge map of the four. Sobel and Prewitt look similar (both combine smoothing and differencing) but
differ subtly in how much they emphasize the center row/column. Kirsch, by taking a *maximum* across
8 compass rotations, can respond even to edges at odd angles that a simple horizontal/vertical pair
of kernels might under-estimate.

✏️ **Your turn.** Add Gaussian noise of a higher sigma (e.g. `add_gaussian_noise(filament_image,
0.1, rng)`) and recompute all four operators. Which one degrades the most visibly? Relate your
answer to whether each operator includes a built-in smoothing step.

### 📘 Lecture notes — edge operators

**Definition.** Roberts, Sobel, Prewitt, and Kirsch are all small, fixed convolution-kernel
approximations to the image gradient; they differ in kernel size, in whether/how they combine
smoothing with differentiation, and (for Kirsch) in using a non-linear maximum over multiple
orientations rather than a simple $(g_x, g_y)$ pair.

**Worked example.** The Sobel kernel in the $x$-direction, $\begin{pmatrix}-1&0&1\\-2&0&2\\-1&0&1
\end{pmatrix}$, factors exactly into $\begin{pmatrix}1\\2\\1\end{pmatrix} \cdot (-1\;0\;1)$ — an
averaging (smoothing) kernel in $y$ combined with a differencing kernel in $x$, which is precisely
why Sobel is more noise-robust than the un-smoothed Roberts operator.

**One memorable point.** *An edge operator with no built-in smoothing (Roberts) will always be more
noise-sensitive than one that combines smoothing and differentiation (Sobel, Prewitt) — this is the
same noise/detail trade-off from Module 03, just embedded directly inside the gradient kernel
itself.*

---
## 3 · The full Canny pipeline

The Canny edge detector (1986) remains one of the most widely used edge detectors precisely
because it assembles the ideas above into a complete, principled three-step procedure: **(1) noise
suppression** (Gaussian smoothing before computing any gradient — recall Module 03's point that
derivatives massively amplify noise if taken on an unsmoothed image), **(2) edge enhancement**
(compute the gradient — Sobel, here — to get edge strength and direction), and **(3) edge
localization**, which itself has two parts: **non-maximum suppression** (thin the gradient-magnitude
ridges down to single-pixel width by keeping, at each pixel, only the value that is a local maximum
along the gradient direction) and **hysteresis thresholding** (use two thresholds — a low $T_L$ and
a high $T_H$ — keeping strong edges above $T_H$ outright, and weak edges above $T_L$ only if they
connect to a strong edge, which links broken edge fragments while suppressing isolated noise).

In [ ]:
# ── 3.1 run the full Canny pipeline step by step ──
smoothed = blur_image(filament_image, gaussian_psf(1.2))                 # step 1: noise suppression
gx, gy, mag, ori = sobel_gradients(smoothed)                              # step 2: edge enhancement
thinned = non_max_suppression_edges(mag, ori)                             # step 3a: non-max suppression
low, high = np.percentile(thinned[thinned > 0], [60, 85])
edges = hysteresis_threshold(thinned, low=low, high=high)                 # step 3b: hysteresis thresholding

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
show_image(axes[0], smoothed, '1. smoothed')
show_image(axes[1], mag, '2. gradient magnitude', cmap='inferno')
show_image(axes[2], thinned, '3a. non-max suppressed', cmap='inferno')
show_image(axes[3], edges.astype(float), '3b. after hysteresis', cmap='gray')
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** Overlay the final `edges` binary map on top of `filament_clean` (the
noise-free ground truth) and visually confirm the detected edges trace the true filament
boundaries. Since each filament is drawn with a Gaussian cross-section, you should see *two* parallel
edges per filament (one on each side), not a single centerline — an important distinction from line
detection (Section 4) and curvilinear-structure detection (Section 5), which aim to find the
*center* of a line rather than its two bounding edges.

✏️ **Your turn.** Change the `low`/`high` percentile cutoffs (e.g. to `[40, 70]` or `[80, 95]`)
and observe how the final edge map changes. What happens to broken, fragmented edges when the gap
between `low` and `high` is too narrow versus too wide?

### 📘 Lecture notes — the Canny pipeline

**Definition.** Canny edge detection is a three-stage pipeline (noise suppression, edge enhancement,
edge localization) in which edge localization itself involves non-maximum suppression (thinning
broad gradient ridges to one-pixel width) followed by hysteresis thresholding (linking weak edges to
strong ones using two thresholds, to minimize both false positives and edge breakage).

**Worked example.** Non-maximum suppression compares each pixel's gradient magnitude only to its two
neighbors *along the gradient direction* (perpendicular to the edge) — a pixel survives only if its
magnitude is not less than either of those two specific neighbors, which is what produces a clean,
single-pixel-wide ridge instead of a thick blur of "edge-ish" pixels.

**One memorable point.** *Hysteresis thresholding's two-threshold trick — strong edges anchor the
decision, weak-but-connected edges get included — directly targets the "broken, fragmented edge"
failure mode that would otherwise plague simple single-threshold edge detection.*

---
## 4 · The Hough transform for line (and curve) detection

Once you have a set of candidate edge points, how do you decide that some of them form a
*line*? The Hough transform reframes this as a voting problem: parameterize a line using
$\rho = x\cos\theta + y\sin\theta$ (this polar-like form avoids the vertical-line singularity of
$y = ax + b$), and for every edge point, let it "vote" for every $(\rho, \theta)$ pair consistent
with a line through that point passing through that point. Accumulate votes in a 2-D array (the
**accumulator**); a strong peak in the accumulator corresponds to strong evidence that a line with
those parameters exists in the image.

In [ ]:
# ── 4.1 run our Hough transform on a simple synthetic edge image with two straight lines ──
size = 120
line_image = np.zeros((size, size))
rr, cc = np.linspace(10, 110, 150).astype(int), np.linspace(20, 100, 150).astype(int)
line_image[rr, cc] = 1.0  # a diagonal line
rr2 = np.full(150, 70)
cc2 = np.linspace(10, 110, 150).astype(int)
line_image[rr2, cc2] = 1.0  # a horizontal line

acc, rhos, thetas = hough_lines(line_image, n_theta=180, n_rho=200)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
show_image(axes[0], line_image, 'edge points (two lines)')
axes[1].imshow(acc, cmap='inferno', aspect='auto',
               extent=[np.rad2deg(thetas[0]), np.rad2deg(thetas[-1]), rhos[-1], rhos[0]])
axes[1].set_xlabel('theta (degrees)'); axes[1].set_ylabel('rho (pixels)')
axes[1].set_title('Hough accumulator (voting space)')
plt.tight_layout(); plt.show()

peak_idx = np.unravel_index(np.argsort(acc.ravel())[-2:], acc.shape)
for r_i, t_i in zip(*peak_idx):
    print(f'peak at rho={rhos[r_i]:.1f}, theta={np.rad2deg(thetas[t_i]):.1f} deg, votes={acc[r_i, t_i]}')

🔍 **What just happened?** Each of the two synthetic lines produces its own sharp, bright peak
in the accumulator — a direct visualization of the "evidence-gathering" character of the Hough
transform: many individual edge points cast overlapping votes, and only genuine straight-line
structure makes those votes pile up at a single $(\rho,\theta)$. Two separate peaks correctly
correspond to our two separate synthetic lines.

✏️ **Your turn.** Add a third line to `line_image` at a new angle and re-run. Confirm a third
peak appears in the accumulator at the corresponding $(\rho, \theta)$.

### 📘 Lecture notes — the Hough transform

**Definition.** The Hough transform is a voting-based feature-extraction method, generalizable
beyond lines to any shape describable by a small number of parameters (e.g. circles of known radius
but unknown center, or ellipses); it converts a difficult pattern-recognition problem into a simpler
peak-detection problem in a quantized parameter (accumulator) space.

**Worked example.** The Hough transform handles partial occlusion and broken lines gracefully
(missing points simply contribute fewer votes to the same peak, rather than breaking the detection
entirely) and is relatively robust to noise, but for curves with many parameters the voting search
becomes very costly, and other shapes can spuriously generate peaks that look like evidence for a
line that is not really there.

**One memorable point.** *Because biological feature geometry is often irregular and dynamic rather
than the fixed, well-defined shapes common in industrial machine-vision applications, Hough-based
detection sees more limited (though still real) use in bioimage analysis compared to industrial
computer vision.*

---
## 5 · Steger's algorithm for curvilinear structures

Filaments and microtubules are thin, ridge-like (not step-like) structures: intensity rises to a
peak along the filament's centerline and falls off on both sides — closer to a 1-D "roof" profile
than a "step" edge. Steger's algorithm (1998) is designed specifically to find the *center* of such
line-like structures directly on grayscale images, without first needing a binary edge map. The core
idea, in 1-D: at a true line center, the first directional derivative is zero and the second
directional derivative is a (negative) local maximum in magnitude — exactly analogous to finding the
peak of a smooth bump. In 2-D, Steger's algorithm computes the local direction of maximum curvature
from the **Hessian matrix**'s eigenvectors, then solves for the sub-pixel position along that
direction where the first directional derivative vanishes.

In [ ]:
# ── 5.1 a simplified 2D Steger-style curvilinear-structure detector (inline; not in the shared toolkit) ──
def steger_centerline(image, sigma=1.3, threshold_percentile=90):
    """A SIMPLIFIED version of Steger's line detector: at each pixel, estimate the Hessian of the
    Gaussian-smoothed image, take the eigenvector of the most negative eigenvalue (the direction of
    strongest second-derivative curvature, i.e. across the line), and keep the pixel as a candidate
    centerline point if the curvature along that direction is strongly negative (a ridge) and the
    first directional derivative along that same direction is close to zero (a true local extremum,
    not just a steep slope)."""
    smoothed = ndi.gaussian_filter(image, sigma)
    Ixx = ndi.gaussian_filter(image, sigma, order=(2, 0))
    Iyy = ndi.gaussian_filter(image, sigma, order=(0, 2))
    Ixy = ndi.gaussian_filter(image, sigma, order=(1, 1))
    Ix = ndi.gaussian_filter(image, sigma, order=(0, 1))
    Iy = ndi.gaussian_filter(image, sigma, order=(1, 0))

    trace = Ixx + Iyy
    det = Ixx * Iyy - Ixy ** 2
    disc = np.sqrt(np.clip((Ixx - Iyy) ** 2 + 4 * Ixy ** 2, 0, None))
    lambda2 = (trace - disc) / 2  # the more negative eigenvalue: direction of strongest ridge curvature

    # eigenvector direction for lambda2 (the curvature-dominant direction, i.e. ACROSS the ridge)
    with np.errstate(invalid='ignore', divide='ignore'):
        angle = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)
    nx, ny = np.cos(angle), np.sin(angle)
    directional_first_deriv = Ix * nx + Iy * ny   # should be ~0 exactly at the ridge center

    candidate = (lambda2 < 0) & (np.abs(directional_first_deriv) < 0.15 * np.abs(lambda2).max())
    strength = np.clip(-lambda2, 0, None)
    thresh = np.percentile(strength[candidate], threshold_percentile) if candidate.any() else np.inf
    centerline = candidate & (strength > thresh)
    return centerline, strength

centerline, ridge_strength = steger_centerline(filament_image, sigma=1.3)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
show_image(axes[0], filament_image, 'image')
show_image(axes[1], ridge_strength, 'ridge strength (-lambda2)', cmap='inferno')
show_image(axes[2], centerline.astype(float), 'detected centerlines')
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** Compare the detected `centerline` points against `filament_clean` — they
should fall along the *middle* of each filament, in contrast to the Canny edges from Section 3,
which traced the two *boundaries* of each filament. This distinction — boundary detection versus
centerline detection — matters in practice: for measuring a filament's length or curvature, a
centerline is usually exactly what you want, while for measuring a filament's width or outline, edge
detection is the right tool instead.

✏️ **Your turn.** Try `sigma=0.8` and `sigma=2.5` in `steger_centerline`. The lecture notes for
Module 04 discussed scale-selection for edge detection; connect what you observe here (does a
too-small or too-large sigma miss the filaments, or detect spurious noise as centerline?) back to
that same scale-selection trade-off.

### 📘 Lecture notes — Steger's algorithm

**Definition.** Steger's curvilinear-structure detector identifies the sub-pixel center of a
line-like (ridge) structure in a grayscale image by analyzing the local Hessian matrix of
second-order partial derivatives: the eigenvector of the most negative eigenvalue gives the
direction across the ridge, and the ridge center is where the directional first derivative along
that direction vanishes while the directional second derivative is strongly negative.

**Worked example.** Because the algorithm operates directly on grayscale intensities rather than a
pre-binarized edge map, it naturally also provides information useful for edge-point detection (by
exploiting knowledge of the already-computed centerline direction), and an unbalanced intensity
profile (e.g. a filament brighter on one side) can be corrected using a look-up table before
centerline extraction.

**One memorable point.** *Choosing the Gaussian smoothing sigma for Steger's algorithm is itself a
scale-selection problem: the sigma should roughly match the expected width of the line-like
structure being detected, directly echoing the scale-space ideas of Module 04.*

## 📌 Key takeaways

- An edge is a location of sharp intensity change; the gradient's magnitude gives edge strength and its direction gives edge orientation.
- Roberts, Sobel, Prewitt, and Kirsch are different fixed-kernel gradient approximations, trading off noise robustness, computational cost, and orientation sensitivity.
- The Canny pipeline (smooth → gradient → non-max suppression → hysteresis thresholding) remains a gold-standard, principled edge detector.
- The Hough transform turns line (and more general curve) detection into a voting/peak-detection problem, robust to noise and partial occlusion but costly for many-parameter shapes.
- Steger's algorithm detects the sub-pixel CENTER of ridge-like (filament) structures directly from grayscale intensities, which is a fundamentally different goal from edge (boundary) detection.

## ✅ Checkpoint

**1.** Why is the Sobel operator generally preferred over the Roberts cross operator for noisy
images?

<details><summary>Answer</summary>
Because the Sobel kernel factors into an averaging (smoothing) component combined with a
differencing component, giving it built-in noise suppression that the un-smoothed Roberts operator
lacks.
</details>

**2.** What problem does hysteresis thresholding solve that a single fixed threshold cannot?

<details><summary>Answer</summary>
A single threshold either breaks weak-but-real edges into fragments (if set too high) or admits too
much noise as spurious edges (if set too low); hysteresis thresholding uses two thresholds so that
weak edges are kept only when connected to strong, confidently real edges, reducing both failure
modes at once.
</details>

**3.** Why would you use Steger's algorithm rather than Canny edge detection if your goal is to
measure the length of a thin filament?

<details><summary>Answer</summary>
Because Canny finds the filament's two boundary edges, not its centerline, while Steger's algorithm
is specifically designed to find the ridge's sub-pixel center directly — the more natural
representation for measuring a thin filament's path length or curvature.
</details>

## 🧠 Questions & discussion

- The Hough transform's strengths (robust to noise and occlusion) seem to directly complement
  Steger's algorithm's strengths (precise centerline localization). Can you imagine a bioimage
  analysis task that might benefit from combining both?
- Kirsch's compass operator takes a maximum over 8 rotated kernels. What is a potential downside of
  always taking a maximum, compared to, say, an average or a more principled combination?
- In what kind of bioimage would you expect edge detection (Section 3) to be clearly more
  appropriate than line/curve detection (Sections 4–5), and vice versa?

## 🚀 Bonus — going further

- Extend the Hough transform implementation to detect circles of a known radius (hint: for each
  edge point, vote for every possible circle *center* at the given radius, rather than for every
  line through that point).
- Compare your simplified Steger implementation's detected centerlines against `skimage.filters`'s
  ridge-detection functions (e.g. `meijering`, `sato`, or `frangi` filters) on the same synthetic
  filament image.
- Investigate what happens to the Canny pipeline's edges when two filaments cross — does
  non-maximum suppression or hysteresis thresholding behave any differently near the crossing
  point?

## 📝 Lab note

```text
Module: 06 — Edges, Lines, and Curves
Date:
Operator I would reach for first on real data, and why:
Edge detection vs. centerline detection — which does MY hypothetical structure need?
```

## 📚 References

1. Canny, J. (1986). A computational approach to edge detection. *IEEE Transactions on Pattern
   Analysis and Machine Intelligence*, 8(6), 679–698.
2. Roberts, L.G. (1963). *Machine Perception of Three-Dimensional Solids* (PhD thesis). MIT.
3. Sobel, I., Feldman, G. (1968). A 3x3 isotropic gradient operator for image processing
   (unpublished talk, Stanford Artificial Intelligence Project).
4. Hough, P.V.C. (1962). Method and means for recognizing complex patterns. US Patent 3,069,654.
5. Duda, R.O., Hart, P.E. (1972). Use of the Hough transformation to detect lines and curves in
   pictures. *Communications of the ACM*, 15(1), 11–15.
6. Steger, C. (1998). An unbiased detector of curvilinear structures. *IEEE Transactions on Pattern
   Analysis and Machine Intelligence*, 20(2), 113–125.
7. Gonzalez, R.C., Woods, R.E. (2018). *Digital Image Processing* (4th ed.). Pearson, Ch. 10.

## ➡️ Next

With Part B and Part C's classical feature-detection toolbox in hand (points in Module 05, lines and
curves here), Part D turns to classifying whole bioimages — starting with classical,
feature-engineering-based machine learning in Module 07.